In [1]:
import numpy as np

In [40]:
# current state of the parameters
x = 90 # meters
y = 90
z = 90
vx = 0.1 # meters / second
vy = 0.1
vz = 0.1


# r = np.array([x, y, z])
# v = np.array([0, 0, 0])


# a vector describing the object`s currrent state
State = np.array([x, y, z, vx, vy, vz]) # 1x6 // In the Kalman filter, the state vector must be a one-dimensional vector 
# Covariance matrix
P_k1 = np.diag([1, 1, 1, 100, 100, 100])



Matrix **P** represents the filter's uncertainty. Small values for position mean high confidence because the camera measured it directly. Large values for velocity mean high uncertainty because the camera doesn't measure speed, so our initial guess of zero is just a wild guess. This large uncertainty signals the filter to quickly adjust the velocity as soon as it sees the position change.

In [51]:
# r = r_0 + v * del_t
# v = v_0
# the initial moment of time
t = 0
# time difference
delta_t = 1

F = np.array([[1, 0, 0, delta_t, 0, 0],
              [0, 1, 0, 0, delta_t, 0],
              [0, 0, 1, 0, 0, delta_t],
              [0, 0, 0, 1, 0, 0],
              [0, 0, 0, 0, 1, 0],
              [0, 0, 0, 0, 0, 1]])

Q = np.diag([0.1, 0.1, 0.1, 1.0, 1.0, 1.0]) 


# updating the vector state
S_predict = F @ State
# updating the covariance matrix
P_k = F @ P_k1 @ F.T + Q

The Matrix F "Projection matrix" allows the transition of one state to another.
The Matrix Q "Process noise matrix" represents the uncertainty of unexpected accelerations. For example, it shows how much accelerations uncertainty exists between frame.

In [52]:
# A function to retrieve the coordinates of the center of mass.
def get_coordinates_from_camera(x_cam, y_cam, z_cam):
    cm_fcamera = np.array([x_cam, y_cam, z_cam])
    return cm_fcamera

In [53]:
H = np.array([[1, 0, 0, 0, 0, 0],
              [0, 1, 0, 0, 0, 0],
              [0, 0, 1, 0, 0, 0]])

The matrix H (Observation matrix) represents which values the sensor measures. For example, it determines which state components are considered by the sensor. We place a 1 in the matrix H to indicate that a specific state is directly measured. If the sensor measures values in pixels, but the state vector is in meters, we should place the conversion coefficient (scale factor) instead of 1.

In [54]:
R = np.diag([0.2, 0.2, 0.2]) # variance on coordinate axes

The matrix R (Measure noise matrix) represents the variance (uncertainty) of the sensor measurement errros.

In [55]:
Expected = H @ S_predict 
Cov_Expected = H @ P_k @ H.T + R
print(Expected)

[98.56153846 98.56153846 98.56153846]


In [56]:
# 1. Obtaining real measurements from the camera
Z = get_coordinates_from_camera(100, 100, 100) # Вектор 3x1

In [57]:
# 2. Calculate the residual (difference between the camera measurement and the filter prediction)
Y = Z - Expected 
print(Y)

[1.43846154 1.43846154 1.43846154]


In [58]:
# 3. Calculate the Kalman Gain
# Увеличение коэффициента Калмана означает, что мы больше доверяем измерениям 
# (увеличили Q или уменьшили R), а его уменьшение — что мы больше доверяем модели 
# (уменьшили Q или увеличили R).
K = P_k @ H.T @ np.linalg.inv(Cov_Expected)
print(K)

[[0.99802567 0.         0.        ]
 [0.         0.99802567 0.        ]
 [0.         0.         0.99802567]
 [0.98716683 0.         0.        ]
 [0.         0.98716683 0.        ]
 [0.         0.         0.98716683]]


In [59]:
# 4. Update the state vector (correct the prediction)
State = S_predict + K @ Y
print(State)

[99.99716    99.99716    99.99716     1.52000152  1.52000152  1.52000152]


In [60]:
# 5. Update covariance
I = np.eye(6) # Единичная матрица 6x6
P_k = (I - K @ H) @ P_k